In [1]:
import os
import requests
import pandas as pd
import time
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# from financetoolkit import Toolkit #uses FMP API

load_dotenv()

True

ACCESS FINANTIAL DATA ONLY FOR 1 STOCK

In [ ]:
# Access Financial Modeling Prep API for additional data

api_key = os.getenv("FMP_API_KEY")
ticker = "AAPL"

# Get company profile example
url_company_profile = f"https://financialmodelingprep.com/api/v3/profile/{ticker}?apikey={api_key}"
url_income_statement = f"https://financialmodelingprep.com/api/v3/income-statement/{ticker}?limit=20&apikey={api_key}"
response_company_profile = requests.get(url_company_profile)
response_income_statement = requests.get(url_income_statement)
data_company_profile = response_company_profile.json()
data_income_statement = response_income_statement.json()


print(data_company_profile[0]['companyName'])
print(data_company_profile[0]['sector'])
print(data_company_profile[0]['country'])

Apple Inc.
Technology
US


In [4]:
print(data_income_statement[0].keys())
transpose_data = pd.DataFrame.from_dict(data_income_statement[0], orient='index').T
transpose_data.columns

dict_keys(['date', 'symbol', 'reportedCurrency', 'cik', 'fillingDate', 'acceptedDate', 'calendarYear', 'period', 'revenue', 'costOfRevenue', 'grossProfit', 'grossProfitRatio', 'researchAndDevelopmentExpenses', 'generalAndAdministrativeExpenses', 'sellingAndMarketingExpenses', 'sellingGeneralAndAdministrativeExpenses', 'otherExpenses', 'operatingExpenses', 'costAndExpenses', 'interestIncome', 'interestExpense', 'depreciationAndAmortization', 'ebitda', 'ebitdaratio', 'operatingIncome', 'operatingIncomeRatio', 'totalOtherIncomeExpensesNet', 'incomeBeforeTax', 'incomeBeforeTaxRatio', 'incomeTaxExpense', 'netIncome', 'netIncomeRatio', 'eps', 'epsdiluted', 'weightedAverageShsOut', 'weightedAverageShsOutDil', 'link', 'finalLink'])


Index(['date', 'symbol', 'reportedCurrency', 'cik', 'fillingDate',
       'acceptedDate', 'calendarYear', 'period', 'revenue', 'costOfRevenue',
       'grossProfit', 'grossProfitRatio', 'researchAndDevelopmentExpenses',
       'generalAndAdministrativeExpenses', 'sellingAndMarketingExpenses',
       'sellingGeneralAndAdministrativeExpenses', 'otherExpenses',
       'operatingExpenses', 'costAndExpenses', 'interestIncome',
       'interestExpense', 'depreciationAndAmortization', 'ebitda',
       'ebitdaratio', 'operatingIncome', 'operatingIncomeRatio',
       'totalOtherIncomeExpensesNet', 'incomeBeforeTax',
       'incomeBeforeTaxRatio', 'incomeTaxExpense', 'netIncome',
       'netIncomeRatio', 'eps', 'epsdiluted', 'weightedAverageShsOut',
       'weightedAverageShsOutDil', 'link', 'finalLink'],
      dtype='object')

In [5]:
# Extract revenue and net income
records = []
for year in data_income_statement:
    records.append({
        'Date': year['calendarYear'],
        'Revenue': year['revenue'],
        'Net Income': year['netIncome']
    })

# Convert to DataFrame
df = pd.DataFrame(records)

# Sort chronologically
df = df.sort_values('Date')

# Optional: Format large numbers
df['Revenue'] = df['Revenue'] / 1e9  # in billions
df['Net Income'] = df['Net Income'] / 1e9  # in billions

df

,Date,Revenue,Net Income
4,2020,274.515,57.411
3,2021,365.817,94.680
2,2022,394.328,99.803
1,2023,383.285,96.995
0,2024,391.035,93.736


In [6]:
# Get key metrics (last year only)
ratios_url = f"https://financialmodelingprep.com/api/v3/key-metrics/{ticker}?limit=1&apikey={api_key}"
ratios_response = requests.get(ratios_url)
ratios_json = ratios_response.json()

print(f"🔍 Raw ratios for {ticker}:\n", ratios_json)

# Handle empty or invalid responses
if not ratios_json or not isinstance(ratios_json, list):
    print(f"⚠️ No ratios found for {ticker}")
    ratios_data = {}
else:
    ratios_data = ratios_json[0]

🔍 Raw ratios for AAPL:
 [{'symbol': 'AAPL', 'date': '2024-09-28', 'calendarYear': '2024', 'period': 'FY', 'revenuePerShare': 25.484914639368924, 'netIncomePerShare': 6.109054070954992, 'operatingCashFlowPerShare': 7.706965094592383, 'freeCashFlowPerShare': 7.091275991064264, 'cashPerShare': 4.247388013764271, 'bookValuePerShare': 3.711600978715614, 'tangibleBookValuePerShare': 3.711600978715614, 'shareholdersEquityPerShare': 3.711600978715614, 'interestDebtPerShare': 7.759429340208995, 'marketCap': 3495160329570, 'enterpriseValue': 3584276329570, 'peRatio': 37.287278415656736, 'priceToSalesRatio': 8.93822887866815, 'pocfratio': 29.55638142954995, 'pfcfRatio': 32.12256867269569, 'pbRatio': 61.37243774486391, 'ptbRatio': 61.37243774486391, 'evToSales': 9.166126637180815, 'enterpriseValueOverEBITDA': 26.617033362072167, 'evToOperatingCashFlow': 30.30997961650346, 'evToFreeCashFlow': 32.94159686022039, 'earningsYield': 0.026818798327209237, 'freeCashFlowYield': 0.03113076074921754, 'debtTo

In [24]:
ratios_data
ratios_df = pd.DataFrame([ratios_data])
#Transpose
ratios_df = ratios_df.T

ratios_df[:60] #There are a total of 60 metrics available



,0
symbol,AAPL
date,2024-09-28
calendarYear,2024
period,FY
revenuePerShare,25.484915
netIncomePerShare,6.109054
operatingCashFlowPerShare,7.706965
freeCashFlowPerShare,7.091276
cashPerShare,4.247388
bookValuePerShare,3.711601


FUNCTION TO GET FUNADMENTALS OF THE STOCK IN "tickers" LIST AND LOOP TO GO THROUGH THE TICKERS AND STORE DATA IN "fundamentals" LIST.

In [8]:
# Get the S&P 500 ticker list from Wikipedia
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
sp500_table = pd.read_html(url)[0]  # The first table is the one we want

# Extract the ticker symbols
tickers = sp500_table["Symbol"].tolist()

# Fix symbols with periods (e.g., BRK.B -> BRK-B for yfinance / FMP compatibility)
tickers = [t.replace(".", "-") for t in tickers]

print(f"✅ {len(tickers)} tickers loaded.")
print(tickers[:10])  # Show first 10

✅ 503 tickers loaded.
['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A']


In [ ]:
# Define function to get profile + ratios for one ticker
def get_fundamentals(ticker, api_key, year):
    try:
        # Get company profile
        profile_url = f"https://financialmodelingprep.com/api/v3/profile/{ticker}?apikey={api_key}"
        profile_data = requests.get(profile_url).json()[0]

        # Get full key metrics history. If you want only the last year, use limit=1 just afer {ticker}?
        ratios_url = f"https://financialmodelingprep.com/api/v3/key-metrics/{ticker}?apikey={api_key}"
        ratios_list = requests.get(ratios_url).json()

        # Find the entry for the specified year
        ratios_data = next((item for item in ratios_list if item["date"].startswith(str(year))), None)
        if not ratios_data:
            print(f"No key metrics found for {ticker} in {year}")
            return None

        return {
            "ticker": ticker,
            "companyName": profile_data.get("companyName"),
            "sector": profile_data.get("sector"),
            "country": profile_data.get("country"),
            "price": profile_data.get("price"),  # this is current, not from the year
            "peRatio": ratios_data.get("peRatio"),
            "revenuePerShare": ratios_data.get("revenuePerShare"),
            #"revenueGrowthPct": ratios_data.get("revenueGrowthPct"), # there's not data
            "netIncomePerShare": ratios_data.get("netIncomePerShare"),
            "freeCashFlowPerShare": ratios_data.get("freeCashFlowPerShare"),
            "roe": ratios_data.get("roe"),
            "longTermGrowthRate": ratios_data.get("longTermGrowthRate")
        }

    except Exception as e:
        print(f"Error fetching data for {ticker}: {e}")
        return None

In [ ]:
# Loop through tickers and collect data
years = range(2024,2025)  # Adjust as needed: there's available data between 2020 and 2024
fundamentals = []
for year in years:
    for ticker in tickers[:5]:  # Limit to first 10 tickers for testing
        print(f"Fetching {ticker} for {year}...")
        data = get_fundamentals(ticker, api_key, year)
        if data:
            data["year"] = year  # add year as a column
            fundamentals.append(data)
        time.sleep(1)  # avoid hitting API rate limits

# Convert to DataFrame and save
df = pd.DataFrame(fundamentals)
#df.to_csv("fundamentals.csv", index=False)
print("✅ Data saved to fundamentals.csv")

Fetching MMM for 2024...
Fetching AOS for 2024...
Fetching ABT for 2024...
Fetching ABBV for 2024...
Fetching ACN for 2024...
✅ Data saved to fundamentals.csv


In [13]:
df[:20]

,ticker,companyName,sector,country,price,peRatio,revenuePerShare,netIncomePerShare,freeCashFlowPerShare,roe,longTermGrowthRate,year
0,MMM,3M Company,Industrials,US,150.09,17.038766,44.616921,7.576253,1.158315,1.086153,None,2024
1,AOS,A. O. Smith Corporation,Industrials,US,70.32,18.790602,25.973999,3.630006,3.223195,0.283302,None,2024
2,ABT,Abbott Laboratories,Healthcare,US,130.69,14.615159,24.224778,7.739225,3.667499,0.281177,None,2024
3,ABBV,AbbVie Inc.,Healthcare,US,198.55,73.480902,31.845110,2.418315,10.080271,1.286617,None,2024
4,ACN,Accenture plc,Technology,IE,247.07,29.552736,103.362513,11.570841,13.720597,0.256809,None,2024


In [10]:
fundamentals_2021 = get_fundamentals("AAPL", api_key, 2023)
print(fundamentals_2021)

{'ticker': 'AAPL', 'companyName': 'Apple Inc.', 'sector': 'Technology', 'country': 'US', 'price': 202.92, 'peRatio': 27.790811789370586, 'revenuePerShare': 24.344472588086393, 'netIncomePerShare': 6.160669263554378, 'freeCashFlowPerShare': 6.325110448392176, 'roe': 1.5607601454639075}


In [ ]:
# Train a simple linear regression model to predict the price 
# Prepare data for modeling

df_clean = df.drop(columns=["ticker","companyName","country","peRatio","revenueGrowthPct"])
df_clean = df_clean.dropna()  # Remove rows with missing values
df_clean = pd.get_dummies(df_clean)

KeyError: "['revenueGrowthPct'] not found in axis"

In [20]:
df_clean[0:10]

,marketCap,price,revenuePerShare,netIncomePerShare,freeCashFlowPerShare,roe,sector_Basic Materials,sector_Communication Services,sector_Consumer Cyclical,sector_Consumer Defensive,sector_Energy,sector_Financial Services,sector_Healthcare,sector_Industrials,sector_Real Estate,sector_Technology,sector_Utilities
0,79841237000,149.900,44.616921,7.576253,1.158315,1.086153,False,False,False,False,False,False,False,True,False,False,False
1,9788051862,69.850,25.973999,3.630006,3.223195,0.283302,False,False,False,False,False,False,False,True,False,False,False
2,227043007000,130.450,24.224778,7.739225,3.667499,0.281177,False,False,False,False,False,False,True,False,False,False,False
3,349190784000,197.685,31.845110,2.418315,10.080271,1.286617,False,False,False,False,False,False,True,False,False,False,False
4,154669971225,248.325,103.362513,11.570841,13.720597,0.256809,False,False,False,False,False,False,False,False,False,True,False
5,144206790000,339.950,48.098859,12.435697,17.499441,0.394186,False,False,False,False,False,False,False,False,False,True,False
6,286550022000,176.730,15.916667,1.012963,1.484568,0.028505,False,False,False,False,False,False,False,False,False,True,False
7,9050155500,12.710,17.390935,2.378187,-6.572238,0.460757,False,False,False,False,False,False,False,False,False,False,True
8,53545480800,99.040,34.285776,9.756246,4.852133,0.208560,False,False,False,False,False,True,False,False,False,False,False
9,32592197775,114.735,22.448276,4.444828,4.734483,0.218549,False,False,False,False,False,False,True,False,False,False,False


In [ ]:
# Split into features and target variable
features = df_clean.drop(columns=["price", "marketCap"])
target = df_clean["price"]
# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)
# Train the model
model = LinearRegression()
model.fit(X_train, y_train)
# Evaluate the model
score = model.score(X_test, y_test)
print(f"Model R^2 score: {score:.4f}")

Model R^2 score: 0.7693


In [19]:
# Make predictions on the test set
predictions = model.predict(X_test)

#Compare predictions with actual prices
predictions_df = pd.DataFrame({
    "Actual Price": y_test,
    "Predicted Price": predictions
})

predictions_df

,Actual Price,Predicted Price
13,67.5700,69.974321
39,179.4950,221.157383
30,505.2200,505.074927
45,188.7700,192.173256
17,66.1336,117.016906
48,299.5700,187.555435
26,297.6200,226.052345
25,113.5950,156.331321
32,298.4600,194.466473
19,195.0399,144.410156
